In [2]:
import os

In [3]:
%pwd

'd:\\kidney_tumor_classification\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'd:\\kidney_tumor_classification'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    training_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list
    params_fine_tune: bool

In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directories
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
        
    def get_training_config (self)->TrainingConfig:
            training = self.config.training
            prepare_base_model =self.config.prepare_base_model
            params =self.params
            training_data =os.path.join(self.config.data_ingestion.unzip_dir,"Kidney ct scan images")
            create_directories([
                Path(training.root_dir)
            ])
            
            training_config =TrainingConfig(
                root_dir=Path(training.root_dir),
                training_model_path=Path(training.training_model_path),
                updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
                training_data=Path(training_data),
                params_epochs=params.EPOCHS,
                params_batch_size=params.BATCH_SIZE,
                params_is_augmentation=params.AUGMENTATION,
                params_image_size=params.IMAGE_SIZE,
                params_fine_tune=params.FINE_TUNE
            )
            
            return training_config

In [9]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import numpy as np
import math
import time

In [ ]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        self.model.summary() 

    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            preprocessing_function=preprocess_input,  #Fixed: use VGG16 preprocessing
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def calculate_class_weights(self):
        labels = self.train_generator.classes
        class_weights = compute_class_weight(
            class_weight="balanced",
            classes=np.unique(labels),
            y=labels
        )
        return dict(enumerate(class_weights))

    def fine_tune_model(self):
        for layer in self.model.layers[:-4]:
            layer.trainable = False

        for layer in self.model.layers[-4:]:
            layer.trainable = True

        self.model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
            loss="categorical_crossentropy",
            metrics=["accuracy"]
    )


    def train(self):
        self.steps_per_epoch = math.ceil(self.train_generator.samples / self.train_generator.batch_size)
        self.validation_steps = math.ceil(self.valid_generator.samples / self.valid_generator.batch_size)

        class_weights = self.calculate_class_weights()  # Handle imbalance

        # Optional: Fine-tune if you've already trained the top layers
        if self.config.params_fine_tune:  # Add this to your config if needed
            self.fine_tune_model()

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_data=self.valid_generator,
            validation_steps=self.validation_steps,
            class_weight=class_weights  # Add class weights
        )

        self.save_model(
            path=self.config.training_model_path,
            model=self.model
        )


In [11]:
try:
    config = ConfigurationManager()
    training_config =config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
    raise e    

[2025-04-21 13:54:38,470: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-04-21 13:54:38,476: INFO: common: yaml file: params.yaml loaded successfully]
[2025-04-21 13:54:38,476: INFO: common: created directory at: artifacts]
[2025-04-21 13:54:38,476: INFO: common: created directory at: artifacts\training]
Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                             

KeyboardInterrupt: 